# 04 — Segmentation & Clustering
Cluster customers, SKUs, and routes into operationally distinct groups.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.models import cluster_customers, cluster_routes

sns.set_theme(style='whitegrid')
%matplotlib inline

PROCESSED = Path('..') / 'data' / 'processed'
orders = pd.read_csv(PROCESSED / 'orders_clean.csv',
                     parse_dates=['order_date','promised_date','actual_delivery_date'])
routes = pd.read_csv(PROCESSED / 'routes_clean.csv', parse_dates=['date'])


## 4.1 Build customer-level aggregates

In [ ]:
if 'customer_id' in orders.columns:
    customer_agg = orders.groupby('customer_id').agg(
        order_frequency=('order_id', 'count'),
        avg_order_value=('shipping_cost', 'mean'),
        avg_distance_km=('distance_km', 'mean') if 'distance_km' in orders.columns else ('shipping_cost', 'mean'),
    ).reset_index()
    print(customer_agg.describe())
    customer_df, km_model, scaler = cluster_customers(customer_agg)
    print("\nCluster sizes:")
    print(customer_df['customer_cluster'].value_counts().sort_index())
else:
    print("'customer_id' column not found. Build customer_agg from your dataset.")
    customer_df = pd.DataFrame()


## 4.2 Visualise customer clusters

In [ ]:
if not customer_df.empty and 'order_frequency' in customer_df.columns:
    fig, ax = plt.subplots(figsize=(9,6))
    scatter = ax.scatter(
        customer_df['order_frequency'],
        customer_df['avg_order_value'],
        c=customer_df['customer_cluster'].astype(float),
        cmap='tab10', alpha=0.7, edgecolors='k', linewidths=0.4
    )
    plt.colorbar(scatter, ax=ax, label='Cluster')
    ax.set_xlabel('Order Frequency')
    ax.set_ylabel('Avg Order Value')
    ax.set_title('Customer Clusters')
    plt.tight_layout()
    plt.savefig('../outputs/figures/customer_clusters.png', dpi=150)
    plt.show()


## 4.3 Route clustering

In [ ]:
route_features = [c for c in [
    'total_distance', 'stop_count', 'vehicle_utilization',
    'route_distance_per_stop', 'avg_speed_kmh'
] if c in routes.columns]

print("Route features available:", route_features)

if route_features:
    routes_clustered, rk_model, r_scaler = cluster_routes(routes, features=route_features)
    print("\nRoute cluster sizes:")
    print(routes_clustered['route_cluster'].value_counts().sort_index())
else:
    print("Not enough route features. Enrich routes_clean.csv first (see src/features.py).")


## 4.4 Silhouette interpretation

In [ ]:
print("""
Silhouette score interpretation
  0.71–1.00  Strong clustering structure
  0.51–0.70  Reasonable structure
  0.26–0.50  Weak structure — consider fewer/more clusters
  ≤ 0.25     No substantial structure found
""")
